In [2]:
from pyspark.sql import SparkSession
from datetime import datetime

In [3]:
sp =  SparkSession.builder.appName('DodgyDeliveries').getOrCreate()

In [4]:
dfDeliLogi = sp.read.csv("Delivery_Logistics.csv" , header="true", inferSchema="true")

In [5]:
dfDeliLogi= dfDeliLogi.na.drop(thresh= 3)
# This will drop any tuple with null values more than 3 attributes

In [6]:
dfDeliLogi.columns

['delivery_id',
 'delivery_partner',
 'package_type',
 'vehicle_type',
 'delivery_mode',
 'region',
 'weather_condition',
 'distance_km',
 'package_weight_kg',
 'delivery_time_hours',
 'expected_time_hours',
 'delayed',
 'delivery_status',
 'delivery_rating',
 'delivery_cost']

In [7]:
dfDeliLogi = dfDeliLogi.drop('delivery_id' , 'delivery_time_hours' , 'expected_time_hours')
# all of this data is bogus 

In [8]:
dfDeliLogi.show()

+----------------+----------------+------------+-------------+-------+-----------------+-----------+-----------------+-------+---------------+---------------+------------------+
|delivery_partner|    package_type|vehicle_type|delivery_mode| region|weather_condition|distance_km|package_weight_kg|delayed|delivery_status|delivery_rating|     delivery_cost|
+----------------+----------------+------------+-------------+-------+-----------------+-----------+-----------------+-------+---------------+---------------+------------------+
|       delhivery|automobile parts|        bike|     same day|   west|            clear|      297.0|            46.96|     no|      delivered|              3|1632.7205999999999|
|      xpressbees|       cosmetics|      ev van|      express|central|             cold|       89.6|            47.39|     no|      delivered|              5|            640.17|
|       shadowfax|       groceries|       truck|      two day|   east|            rainy|      273.5|          

Lable encoding feature delayed into delayedNUM to get a operable integer value


In [9]:
dfDeliLogi.groupBy('weather_condition').avg().show()
dfDeliLogi.groupBy('region').avg().show()
dfDeliLogi.groupBy('weather_condition').avg().show()

+-----------------+------------------+----------------------+--------------------+------------------+
|weather_condition|  avg(distance_km)|avg(package_weight_kg)|avg(delivery_rating)|avg(delivery_cost)|
+-----------------+------------------+----------------------+--------------------+------------------+
|            rainy| 150.9802685207381|    25.249470151042964|  3.4725485495085113| 867.4693553584276|
|              hot|150.89445520581117|     25.29713801452787|  3.8627118644067795| 868.2310804842624|
|           stormy| 149.2663649356832|    25.180678894711825|   3.354454502143878|  859.247482039066|
|             cold| 150.7004569504567|    25.163468013468016|   3.877104377104377|  866.776034247235|
|            clear|150.48307468477154|    24.750545586808926|  3.8358389912706112| 865.3669725509246|
|            foggy|150.03631192225643|    25.229978667930773|  3.6006162597771985| 862.6822355060455|
+-----------------+------------------+----------------------+--------------------+

In [20]:

# 9/7/26 
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol="vehicle_type",outputCol="vehicleLE")
indexedDF= indexer.fit(dfDeliLogi).transform(dfDeliLogi)
indexedDF.show()

# indexer = StringIndexer(inputCol="City", outputCol="City_label") 
# indexed_df = indexer.fit(dfs).transform(dfs)
# indexed_df=indexed_df.drop("City").withColumnRenamed("City_label","City")
# print("The output dataframe is:")
# indexed_df.show()
# spark.sparkContext.stop()

+----------------+----------------+------------+-------------+-------+-----------------+-----------+-----------------+-------+---------------+---------------+------------------+---------+
|delivery_partner|    package_type|vehicle_type|delivery_mode| region|weather_condition|distance_km|package_weight_kg|delayed|delivery_status|delivery_rating|     delivery_cost|vehicleLE|
+----------------+----------------+------------+-------------+-------+-----------------+-----------+-----------------+-------+---------------+---------------+------------------+---------+
|       delhivery|automobile parts|        bike|     same day|   west|            clear|      297.0|            46.96|     no|      delivered|              3|1632.7205999999999|      3.0|
|      xpressbees|       cosmetics|      ev van|      express|central|             cold|       89.6|            47.39|     no|      delivered|              5|            640.17|      5.0|
|       shadowfax|       groceries|       truck|      two da

In [21]:
indexedDF.select('vehicleLE','vehicle_type').show()


+---------+------------+
|vehicleLE|vehicle_type|
+---------+------------+
|      3.0|        bike|
|      5.0|      ev van|
|      4.0|       truck|
|      5.0|      ev van|
|      1.0|         van|
|      0.0|     ev bike|
|      2.0|     scooter|
|      1.0|         van|
|      1.0|         van|
|      4.0|       truck|
|      2.0|     scooter|
|      4.0|       truck|
|      5.0|      ev van|
|      3.0|        bike|
|      0.0|     ev bike|
|      5.0|      ev van|
|      3.0|        bike|
|      0.0|     ev bike|
|      1.0|         van|
|      4.0|       truck|
+---------+------------+
only showing top 20 rows


alright , see now i gotta learn these pipeline 

StringIndexer (all categoricals)
      ↓
OneHotEncoder
      ↓
VectorAssembler (combine OHE + raw numerics into one vector)
      ↓
StandardScaler (scale the assembled vector)
      ↓
RandomForestClassifier


- search on scholar.google.com for accurate statistics

Lable Encoding of some features (Vehicle Types,region, Weather)

- vehicle ['van','ev bike','ev van' , 'truck' , '
scooter' , 'bike' ]
- region ['central','west','east','north','south']
- Weather Conditions ['rainy', 'hot','stormy','cold','clear','foggy']



REASON OF GETTING LATE (refer featuresML.drawio)
- Distance
- Package Weight
- Weather Conditions
- Region
- Vehicle Type